In [1]:
import torch
import torch.nn as nn
from collections import OrderedDict

def forward_in_p_steps_general(model: nn.Module, x: torch.Tensor, p: int, 
                               desired: list[str] | None = None, 
                               keep_all: bool = False):
    """
    Run any model in p steps, collect intermediate states.

    Args:
        model: nn.Module (any model)
        x: input tensor
        p: number of steps to divide the model into
        desired: list of layer names to keep (optional)
        keep_all: if True, keep all intermediate activations (not just last in each step)

    Returns:
        states: dict with "input", "step_k", "output", and/or specific layers
    """

    # --- get ordered list of submodules (skip root) ---
    layers = OrderedDict()
    for name, module in model.named_modules():
        if name != "":
            layers[name] = module

    layer_names = list(layers.keys())
    n = len(layer_names)
    chunk_size = (n + p - 1) // p   # ceil division

    states = {"input": x}
    outputs = {}

    # --- hook function ---
    def get_hook(name):
        def hook(_, __, output):
            outputs[name] = output.detach()
        return hook

    # register hooks
    handles = []
    for name, module in layers.items():
        handles.append(module.register_forward_hook(get_hook(name)))

    # run forward pass
    out = model(x)

    # remove hooks
    for h in handles:
        h.remove()

    # always store final output
    states["output"] = out

    # --- decide what to keep ---
    if desired is not None:  # explicit selection
        for name in desired:
            if name in outputs:
                states[name] = outputs[name]
    elif keep_all:  # keep everything
        states.update(outputs)
    else:  # default: just last in each chunk
        for step in range(p):
            start = step * chunk_size
            end   = min((step + 1) * chunk_size, n)
            if start >= end:
                continue
            last_name = layer_names[end - 1]
            states[f"step_{step+1}"] = outputs[last_name]

    return states


In [2]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(5, 16)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(16, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        return self.fc3(x)

model = MyModel()
x = torch.randn(512, 5)

# Case 1: split into 2 steps (just states per chunk)
states = forward_in_p_steps_general(model, x, p=2)
print(states.keys())
# -> ['input', 'step_1', 'step_2', 'output']

# Case 2: keep all activations
states_all = forward_in_p_steps_general(model, x, p=2, keep_all=True)
print(states_all.keys())
# -> ['input', 'fc1', 'relu1', 'fc2', 'relu2', 'fc3', 'output']

# Case 3: keep only specific layers
states_sel = forward_in_p_steps_general(model, x, p=3, desired=["fc2", "relu2"])
print(states_sel.keys())
# -> ['input', 'output', 'fc2', 'relu2']


dict_keys(['input', 'output', 'step_1', 'step_2'])
dict_keys(['input', 'output', 'fc1', 'relu1', 'fc2', 'relu2', 'fc3'])
dict_keys(['input', 'output', 'fc2', 'relu2'])


In [5]:
print(states_sel['input'])

tensor([[-0.9015,  0.4544,  2.2436, -0.4493, -0.0186],
        [-1.1389,  0.5593, -0.7414, -1.1220,  0.8957],
        [-0.3406, -1.2966,  0.4512, -1.1335, -0.1355],
        ...,
        [-0.6052,  0.2502,  0.4810,  2.2268,  1.3144],
        [ 1.3267,  0.4136,  0.8465, -1.2394, -0.9914],
        [-0.2759, -1.3738, -0.6636, -0.1518, -0.3383]])
